# Analyze Actions & Memory

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from env.conversation_env import ConversationEnv
from stable_baselines3 import PPO

## Run Policy

In [ ]:
env = ConversationEnv()

model = PPO.load("ppo_memory_agent")
obs, _ = env.reset()
done = False

while not done:
    action, _ = model.predict(
        obs,
        deterministic=True
    )
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

## Save Events in DataFrame

In [ ]:
events = []

for mem_id, e in env.memory_events.items():
    events.append({
        "id": e.memory_id,
        "created": e.created_step,
        "deleted": e.deleted_step,
        "lifetime": (e.deleted_step - e.created_step)
            if e.deleted_step is not None else None,
        "action": e.action,
        "replaced_by": e.replaced_by,
        "content": e.content,
    })

df = pd.DataFrame(events)
df.head()

## Memory Lifetime

In [ ]:
plt.figure(figsize=(6,4))

df["lifetime"].dropna().hist(bins=20)

plt.title("Memory Lifetime Distribution")
plt.xlabel("Steps alive")
plt.ylabel("Count")
plt.show()

## Memory Survival

In [ ]:
max_step = df["deleted"].dropna().max()

survival = []

for t in range(int(max_step) + 1):
    alive = df[
        (df["created"] <= t) &
        ((df["deleted"].isna()) | (df["deleted"] > t))
    ].shape[0]

    survival.append(alive)

plt.plot(survival)
plt.title("Memory Survival Curve")
plt.xlabel("Step")
plt.ylabel("Active Memories")
plt.show()

In [ ]:
df.groupby("action")["lifetime"].mean().plot(kind="bar")

plt.title("Average Memory Lifetime by Action")
plt.ylabel("Lifetime")
plt.show()

In [ ]:
replacements = df[df["replaced_by"].notna()]
replacements[["id", "replaced_by", "content"]]

In [ ]:
import networkx as nx

G = nx.DiGraph()

for _, row in replacements.iterrows():
    G.add_edge(row["id"], row["replaced_by"])

plt.figure(figsize=(6,4))
nx.draw(G, with_labels=True, node_size=800)
plt.title("Memory Replacement Graph")
plt.show()

In [ ]:
forgotten = df[df["lifetime"] < 2]
forgotten

In [ ]:
forgotten["content"].value_counts().head(10)

## Memory Efficiency vs. Correctness

In [ ]:
plt.scatter(
    env.global_step,
    [info.get("correct", 0)],
)

plt.title("Correctness over time")
plt.show()